<< [Sudoku-4-Z3](Sudoku-4-Z3.ipynb) | [Index](README.md) | [Sudoku-6-Infer](Sudoku-6-Infer.ipynb) >>

# Resolution de Sudoku avec Algorithm X et Dancing Links

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Comprendre le probleme de la couverture exacte et sa formulation matricielle
2. Implementer l'algorithme X de Knuth avec la technique Dancing Links (DLX)
3. Modéliser le Sudoku comme un probleme de couverture exacte (729 lignes, 324 colonnes)
4. Comparer une approche bibliotheque (DlxLib) et une implementation from scratch

### Prerequis
- Sudoku-0-Environment (classes de base)
- Comprehension du backtracking et des algorithmes recursifs
- Notions de structures de donnees (listes chaînees, matrices creuses)

### Duree estimee : 45 minutes

## 1. Introduction a Dancing Links

Dancing Links (DLX) est une technique efficace pour resoudre des problemes de couverture exacte, popularisee par Donald Knuth. Elle est souvent utilisee pour des problemes comme le Sudoku, ou l'objectif est de couvrir toutes les contraintes avec un ensemble de solutions possibles.

**References** :
- [Dancing Links Algorithm](https://en.wikipedia.org/wiki/Dancing_Links)
- [DLXLib Documentation](https://github.com/tomerfiliba/dlx)

### Theorie de la Couverture Exacte

Le probleme de la couverture exacte consiste a couvrir un ensemble d'elements, chaque element etant couvert par exactement un sous-ensemble. Cela est utile pour des problemes tels que le pavage d'un echiquier avec des pentominos, le probleme des huit reines, et la resolution de Sudoku. Le probleme est NP-complet et peut etre resolu par l'algorithme X de Donald Knuth, souvent implemente avec la technique des Dancing Links (DLX).
- [Problemes de couverture exacte](https://fr.wikipedia.org/wiki/Probleme_de_la_couverture_exacte).

#### Exemples

Soit un ensemble U = {0, 1, 2, 3, 4} et une collection de sous-ensembles S = {E, I, P} avec E = {0, 2, 4}, I = {1, 3} et P = {2, 3}. Une couverture exacte de U est une sous-collection de S où chaque element de U est contenu dans exactement un sous-ensemble de cette sous-collection, par exemple {E, I}.

#### Representation Matricielle

Le probleme de la couverture exacte peut etre represente par une matrice où chaque ligne represente un sous-ensemble et chaque colonne represente un element. Une entree de la matrice est 1 si l'element de la colonne est dans le sous-ensemble de la ligne, et 0 sinon. Une couverture exacte est une selection de lignes telle que chaque colonne contient exactement un 1.

Pour le Sudoku, la matrice contient 729 lignes (pour chaque cellule et chaque valeur possible) et 324 colonnes (pour les contraintes de ligne-colonne, ligne-nombre, colonne-nombre, et boîte-nombre).


## 2. Configuration de l'environnement

Installez les packages necessaires pour ce notebook :


In [1]:
#r "nuget: DlxLib"

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages DlxLib, 1.3.0

### Importation des Classes de Base

Nous allons importer les classes de base définies dans le notebook précédent, fournissant notamment la représentation, le chargement et l'affichage de Sudokus, et l'infrastructure de résolution.


In [2]:
#!import Sudoku-0-Environment-Csharp.ipynb

Error: System.IO.FileNotFoundException: Could not find file 'C:\dev\CoursIA\Sudoku-0-Environment-Csharp.ipynb'.
File name: 'C:\dev\CoursIA\Sudoku-0-Environment-Csharp.ipynb'
   at Microsoft.Win32.SafeHandles.SafeFileHandle.CreateFile(String fullPath, FileMode mode, FileAccess access, FileShare share, FileOptions options)
   at System.IO.Strategies.OSFileStreamStrategy..ctor(String path, FileMode mode, FileAccess access, FileShare share, FileOptions options, Int64 preallocationSize, Nullable`1 unixCreateMode)
   at System.IO.Strategies.AsyncWindowsFileStreamStrategy..ctor(String path, FileMode mode, FileAccess access, FileShare share, FileOptions options, Int64 preallocationSize, Nullable`1 unixCreateMode)
   at System.IO.Strategies.FileStreamHelpers.ChooseStrategyCore(String path, FileMode mode, FileAccess access, FileShare share, FileOptions options, Int64 preallocationSize, Nullable`1 unixCreateMode)
   at System.IO.FileStream..ctor(String path, FileMode mode, FileAccess access

## 3. Implémentation avec DLXLib

Nous allons commencer par implémenter le solver en utilisant la bibliothèque `DlxLib`.

In [3]:
using System;
using System.Collections.Generic;
using System.Collections.Immutable;
using DlxLib;

public class DancingLinkSolver : ISudokuSolver
{
     // Méthode principale pour résoudre un Sudoku
    public SudokuGrid Solve(SudokuGrid s)
    {
        // Conversion de la grille de Sudoku en une liste de tuples représentant les contraintes internes
        var internalRows = BuildInternalRowsForGrid(s);

        // Conversion des contraintes internes en lignes compatibles avec DLX
        var dlxRows = BuildDlxRows(internalRows);

        // Résolution du problème de couverture exacte avec DLX
        var solutions = new Dlx()
            .Solve(dlxRows, d => d, r => r)
            .ToImmutableList();

         // Conversion de la solution trouvée en une grille de Sudoku
        return SolutionToGrid(internalRows, solutions.First());
    }
    
    // Construction des contraintes internes pour chaque cellule de la grille
    private static IImmutableList<Tuple<int, int, int, bool>> BuildInternalRowsForGrid(SudokuGrid grid)
    {
        var internalRows = new List<Tuple<int, int, int, bool>>();
        for (int row = 0; row < 9; row++)
        {
            for (int col = 0; col < 9; col++)
            {
                int value = grid.Cells[row, col];
                internalRows.AddRange(BuildInternalRowsForCell(row, col, value));
            }
        }
        return internalRows.ToImmutableList();
    }

    // Construction des contraintes internes pour une cellule spécifique
    private static IImmutableList<Tuple<int, int, int, bool>> BuildInternalRowsForCell(int row, int col, int value)
    {
        if (value >= 1 && value <= 9)
        {
            return ImmutableList.Create(Tuple.Create(row, col, value, true));
        }
        else
        {
            var internalRows = new List<Tuple<int, int, int, bool>>(9);
            for (int v = 1; v <= 9; v++)
            {
                internalRows.Add(Tuple.Create(row, col, v, false));
            }
            return internalRows.ToImmutableList();
        }
    }

    // Conversion des contraintes internes en lignes pour DLX
    private static IImmutableList<IImmutableList<int>> BuildDlxRows(IEnumerable<Tuple<int, int, int, bool>> internalRows)
    {
        var dlxRows = new List<IImmutableList<int>>();
        foreach (var internalRow in internalRows)
        {
            dlxRows.Add(BuildDlxRow(internalRow));
        }
        return dlxRows.ToImmutableList();
    }

    // Construction d'une ligne DLX à partir d'une contrainte interne
    private static IImmutableList<int> BuildDlxRow(Tuple<int, int, int, bool> internalRow)
    {
        var row = internalRow.Item1;
        var col = internalRow.Item2;
        var value = internalRow.Item3;
        var box = RowColToBox(row, col);
        var result = new int[4 * 9 * 9];

        // Chaque contrainte est représentée par un 1 dans les colonnes appropriées
        result[row * 9 + col] = 1; 
        result[9 * 9 + row * 9 + value - 1] = 1;
        result[2 * 9 * 9 + col * 9 + value - 1] = 1;
        result[3 * 9 * 9 + box * 9 + value - 1] = 1;
        return result.ToImmutableList();
    }

    // Conversion des coordonnées de ligne et de colonne en un index de boîte
    private static int RowColToBox(int row, int col)
    {
        return row - (row % 3) + (col / 3);
    }

    // Conversion de la solution DLX en une grille de Sudoku
    private static SudokuGrid SolutionToGrid(
        IReadOnlyList<Tuple<int, int, int, bool>> internalRows,
        Solution solution)
    {
        var grid = new int[9, 9];
        foreach (var (row, col, value, _) in solution.RowIndexes.Select(rowIndex => internalRows[rowIndex]))
        {
            grid[row, col] = value;
        }
        return new SudokuGrid { Cells = grid };
    }
}


(6,34): error CS0246: Le nom de type ou d'espace de noms 'ISudokuSolver' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)

(9,29): error CS0246: Le nom de type ou d'espace de noms 'SudokuGrid' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)

(9,12): error CS0246: Le nom de type ou d'espace de noms 'SudokuGrid' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)

(27,88): error CS0246: Le nom de type ou d'espace de noms 'SudokuGrid' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)

(94,20): error CS0246: Le nom de type ou d'espace de noms 'SudokuGrid' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)

(103,20): error CS0246: Le nom de type ou d'espace de noms 'SudokuGrid' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)



Error: compilation error

### Analyse du solveur DLXLib

La classe `DancingLinkSolver` implemente un solveur Sudoku utilisant la bibliotheque DlxLib.

| Methode | Responsabilité |
|---------|----------------|
| **BuildInternalRowsForGrid** | Convertit la grille en tuples (row, col, value, isFixed) |
| **BuildDlxRow** | Cree une ligne DLX avec 4 contraintes (cell, row, col, box) |
| **SolutionToGrid** | Convertit la solution DLX en grille Sudoku |

**Points cles** :
1. Chaque cellule vide genere 9 possibilites (valeurs 1-9)
2. Chaque cellule remplie genere 1 seule possibilite
3. La matrice DLX a 4 x 9 x 9 = 324 colonnes de contraintes
4. DlxLib gere automatiquement l'algorithme X et le backtracking

> **Note technique** : L'utilisation d'une bibliotheque externe simplifie considérablement le code mais ajoute une dependance au projet.


## 4. Implémentation Optimisée from Scratch

Nous allons maintenant implémenter une version optimisée de l'algorithme DLX.
Les Dancing Links (DLX) sont une technique pour ajouter et supprimer efficacement des nœuds d'une liste doublement chaînée circulaire. Utilisés pour implémenter l'algorithme X de Knuth, ils permettent de résoudre des problèmes de couverture exacte comme le Sudoku.

### Implémentation de l'Algorithme DLX

L'algorithme X est un algorithme de backtracking récursif qui trouve toutes les solutions au problème de couverture exacte. Pour améliorer l'efficacité, une matrice creuse est utilisée où seuls les 1 sont stockés.

### Fonctionnement

- Chaque nœud de la matrice pointe vers les nœuds adjacents à gauche et à droite (dans la même ligne), en haut et en bas (dans la même colonne), et vers l'en-tête de colonne.
- Chaque colonne a un nœud spécial "en-tête de colonne" inclus dans la liste circulaire des colonnes restantes.
- Lors de l'élimination d'une colonne, les lignes contenant un 1 dans cette colonne sont également éliminées, car elles entrent en conflit.

### Sélection et Couverture

- Sélectionnez une colonne avec le plus petit nombre de 1.
- Pour chaque ligne contenant un 1 dans cette colonne, ajoutez cette ligne à la solution partielle et éliminez les colonnes avec un 1 dans cette ligne.
- Répétez jusqu'à ce que toutes les colonnes soient éliminées, formant ainsi une solution.
- Pour revenir en arrière, restaurez les colonnes et les lignes éliminées dans l'ordre inverse.

In [4]:
public class DlxCustomized
{
    // Classe représentant l'en-tête de colonne dans DLX
    class NodeHead : Node
    {
        internal int size;

        public NodeHead() : base(null) { }
    }

    // Classe représentant un nœud dans la structure DLX
    class Node
    {
        internal Node right = null;
        internal Node left = null;
        internal Node up = null;
        internal Node down = null;
        internal NodeHead nodeHead = null;
        
        internal int rowIndex;
        internal int column ;
        
        internal int value;

        public Node(NodeHead t)
        {
            nodeHead = t;
        }
    }
    
    private NodeHead root;
    private bool stop = false;
    private LinkedList<Node> solutions = new LinkedList<Node>();
    private SudokuGrid sudoku;
    
    // Constructeur initialisant la grille de Sudoku
    public DlxCustomized(SudokuGrid sudokuGrid)
    {
        sudoku = sudokuGrid;
        root = new NodeHead();
        root.right = root;
        root.left = root;
    }

    // Méthode principale pour résoudre le Sudoku
    public SudokuGrid Solve()
    {
        Init();
        search();
        foreach(Node node in solutions)
        {
            sudoku.Cells[node.rowIndex, node.column] = node.value;
        }
        return sudoku;
    }
    
    // Initialisation de la structure DLX
    public void Init()
    {
        Node[] tmp = new Node[9 * 9 * 4];
        root = new NodeHead();
        Node currentNode = root;
        
        for (int j = 0; j < 324; j++)
        {
            currentNode.right = new NodeHead();
            currentNode.right.left = currentNode;
            currentNode = currentNode.right;
            currentNode.rowIndex = j;
            currentNode.up = currentNode;
            currentNode.down = currentNode;
            tmp[j] = currentNode;
        }
        currentNode.right = root;
        root.left = currentNode;

        // Initialisation des nœuds pour chaque cellule du Sudoku

        int imatrix = 0;
        for (int i = 0; i < 9; i++)
        {
            for (int j = 0; j < 9; j++)
            {
                int value = sudoku.Cells[i ,j];
                Node tmpRCCNode, tmpRNCNode, tmpCNCNode, tmpBNCNode;
                
                if (value == 0)
                {
                    value = 1;
                    // Contrainte de cellule
                    int RCC = 9 * i + j; 
                    // Contrainte de ligne
                    int RNC = 81 + 9 * i + value - 1;
                    // Contrainte de colonne
                    int CNC = 162 + 9 * j + value - 1;
                    // Contrainte de bloc
                    int BNC = 243 + ((i / 3) * 3 + j / 3) * 9 + value - 1;
                    int end = imatrix + 9;
                    for (; imatrix < end; imatrix++)
                    {
                         // Création des nœuds pour les contraintes
                        tmpRCCNode = new Node((NodeHead)tmp[RCC].down);
                        tmpRNCNode = new Node((NodeHead)tmp[RNC].down);
                        tmpCNCNode = new Node((NodeHead)tmp[CNC].down);
                        tmpBNCNode = new Node((NodeHead)tmp[BNC].down);
                        
                        // Initialisation des nœuds avec les indices de ligne et de colonne
                        tmpRCCNode.rowIndex = i;
                        tmpRCCNode.column = j;
                        tmpRCCNode.value = value;
                        tmpRNCNode.rowIndex = i;
                        tmpRNCNode.column = j;
                        tmpRNCNode.value = value;
                        tmpCNCNode.rowIndex = i;
                        tmpCNCNode.column = j;
                        tmpCNCNode.value = value;
                        tmpBNCNode.rowIndex = i;
                        tmpBNCNode.column = j;
                        tmpBNCNode.value = value++;

                        // Mise à jour des tailles des en-têtes de colonne
                        ((NodeHead)tmp[RCC].down).size++;
                        ((NodeHead)tmp[RNC].down).size++;
                        ((NodeHead)tmp[CNC].down).size++;
                        ((NodeHead)tmp[BNC].down).size++;

                        // Liaisons des nœuds entre eux pour former des listes doublement chaînées circulaires
                        tmpRCCNode.right = tmpRNCNode;
                        tmpRNCNode.right = tmpCNCNode;
                        tmpCNCNode.right = tmpBNCNode;
                        tmpBNCNode.right = tmpRCCNode;
                        tmpBNCNode.left = tmpCNCNode;
                        tmpCNCNode.left = tmpRNCNode;
                        tmpRNCNode.left = tmpRCCNode;
                        tmpRCCNode.left = tmpBNCNode;

                        // Liaisons des nœuds avec leurs en-têtes de colonne respectifs
                        tmpRCCNode.up = tmp[RCC];
                        tmpRCCNode.down = tmp[RCC].down;
                        tmp[RCC].down = tmpRCCNode;
                        tmp[RCC] = tmpRCCNode;
                        tmpRNCNode.up = tmp[RNC];
                        tmpRNCNode.down = tmp[RNC].down;
                        tmp[RNC].down = tmpRNCNode;
                        tmp[RNC++] = tmpRNCNode;
                        tmpCNCNode.up = tmp[CNC];
                        tmpCNCNode.down = tmp[CNC].down;
                        tmp[CNC].down = tmpCNCNode;
                        tmp[CNC++] = tmpCNCNode;
                        tmpBNCNode.up = tmp[BNC];
                        tmpBNCNode.down = tmp[BNC].down;
                        tmp[BNC].down = tmpBNCNode;
                        tmp[BNC++] = tmpBNCNode;
                    }
                }
                else
                {
                     // Contrainte de cellule
                    int RCC = 9 * i + j;
                    // Contrainte de ligne
                    int RNC = 81 + 9 * i + value - 1;
                    // Contrainte de colonne
                    int CNC = 162 + 9 * j + value - 1;
                    // Contrainte de bloc
                    int BNC = 243 + ((i / 3) * 3 + j / 3) * 9 + value - 1;

                    // Création des nœuds pour les contraintes
                    tmpRCCNode = new Node((NodeHead)tmp[RCC].down);
                    tmpRNCNode = new Node((NodeHead)tmp[RNC].down);
                    tmpCNCNode = new Node((NodeHead)tmp[CNC].down);
                    tmpBNCNode = new Node((NodeHead)tmp[BNC].down);

                    // Initialisation des nœuds avec les indices de ligne et de colonne
                    tmpRCCNode.rowIndex = i;
                    tmpRCCNode.column = j;
                    tmpRCCNode.value = value;
                    tmpRNCNode.rowIndex = i;
                    tmpRNCNode.column = j;
                    tmpRNCNode.value = value;
                    tmpCNCNode.rowIndex = i;
                    tmpCNCNode.column = j;
                    tmpCNCNode.value = value;
                    tmpBNCNode.rowIndex = i;
                    tmpBNCNode.column = j;
                    tmpBNCNode.value = value;

                    // Liaisons des nœuds entre eux pour former des listes doublement chaînées circulaires
                    tmpRCCNode.right = tmpRNCNode;
                    tmpRNCNode.right = tmpCNCNode;
                    tmpCNCNode.right = tmpBNCNode;
                    tmpBNCNode.right = tmpRCCNode;
                    tmpBNCNode.left = tmpCNCNode;
                    tmpCNCNode.left = tmpRNCNode;
                    tmpRNCNode.left = tmpRCCNode;
                    tmpRCCNode.left = tmpBNCNode;

                     // Liaisons des nœuds avec leurs en-têtes de colonne respectifs
                    tmpRCCNode.up = tmp[RCC];
                    tmpRCCNode.down = tmp[RCC].down;
                    tmp[RCC].down = tmpRCCNode;
                    tmp[RCC] = tmpRCCNode;
                    tmpRNCNode.up = tmp[RNC];
                    tmpRNCNode.down = tmp[RNC].down;
                    tmp[RNC].down = tmpRNCNode;
                    tmp[RNC++] = tmpRNCNode;
                    tmpCNCNode.up = tmp[CNC];
                    tmpCNCNode.down = tmp[CNC].down;
                    tmp[CNC].down = tmpCNCNode;
                    tmp[CNC++] = tmpCNCNode;
                    tmpBNCNode.up = tmp[BNC];
                    tmpBNCNode.down = tmp[BNC].down;
                    tmp[BNC].down = tmpBNCNode;
                    tmp[BNC++] = tmpBNCNode;
                    imatrix++;
                }
            }
        }
    }
    
    // Méthode de recherche récursive
    public void search()
    {
        if (root.right == root)
        {
            stop = true;
            return;
        }

        NodeHead selected = (NodeHead)root.right;
        int c = selected.size;
        for (NodeHead currentNode = (NodeHead)root.right; currentNode != root; currentNode = (NodeHead)currentNode.right)
        {
            if (c > currentNode.size)
            {
                c = currentNode.size;
                selected = currentNode;
            }
        }

        cover(selected);

        for (Node iNode = selected.down; iNode != selected; iNode = iNode.down)
        {
            solutions.AddLast(iNode);
            for (Node jNode = iNode.right; jNode != iNode; jNode = jNode.right)
            {
                cover(jNode.nodeHead);
            }
            search();
            if (stop)
                return;
            solutions.RemoveLast();
            for (Node jNode = iNode.left; jNode != iNode; jNode = jNode.left)
            {
                uncover(jNode.nodeHead);
            }
        }
        uncover(selected);
        return;
    }

    // Méthode pour couvrir une colonne
    private void cover(NodeHead node)
    {
        node.left.right = node.right;
        node.right.left = node.left;
        for (Node iNode = node.down; iNode != node; iNode = iNode.down)
        {
            for (Node jNode = iNode.right; jNode != iNode; jNode = jNode.right)
            {
                jNode.up.down = jNode.down;
                jNode.down.up = jNode.up;
                jNode.nodeHead.size--;
            }
        }
    }

    // Méthode pour découvrir une colonne
    private void uncover(NodeHead node)
    {
        for (Node iNode = node.down; iNode != node; iNode = iNode.down)
        {
            for (Node jNode = iNode.right; jNode != iNode; jNode = jNode.right)
            {
                jNode.up.down = jNode;
                jNode.down.up = jNode;
                jNode.nodeHead.size++;
            }
        }
        node.left.right = node;
        node.right.left = node;
    }
}


(34,13): error CS0246: Le nom de type ou d'espace de noms 'SudokuGrid' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)

(37,26): error CS0246: Le nom de type ou d'espace de noms 'SudokuGrid' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)

(46,12): error CS0246: Le nom de type ou d'espace de noms 'SudokuGrid' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)



Error: compilation error

### Analyse de l'implementation DLX personnalisée

La classe `DlxCustomized` implemente l'algorithme Dancing Links de maniere complete.

| Composant | Description |
|-----------|-------------|
| **Node** | Nœud de la matrice creuse avec 4 pointeurs (left, right, up, down) |
| **NodeHead** | En-tete de colonne avec compteur de taille |
| **Init()** | Construction de la matrice DLX (324 colonnes) |
| **search()** | Algorithme X recursif avec heuristique de colonne minimum |
| **cover/uncover** | Operations de "dancing links" pour backtracking efficace |

**Points cles** :
1. Les 324 colonnes representent les contraintes Sudoku (cell, row, col, box)
2. La methode `cover()` supprime une colonne et ses lignes conflits
3. La methode `uncover()` restaure en ordre inverse pour le backtrack
4. L'heuristique de colonne minimum minimise la profondeur de recherche

> **Note technique** : La structure circulaire des nœuds permet d'ajouter et supprimer des elements en O(1) sans reallocation de memoire.


### Implémentation du nouveau solver

In [5]:
public class DancingLinkSolverWithCustomDlx : ISudokuSolver
{
    
       
    public SudokuGrid Solve(SudokuGrid sudoku)
    {
        DlxCustomized dlxCustomized = new DlxCustomized(sudoku);
        return dlxCustomized.Solve();
    }
}


(1,47): error CS0246: Le nom de type ou d'espace de noms 'ISudokuSolver' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)

(5,29): error CS0246: Le nom de type ou d'espace de noms 'SudokuGrid' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)

(5,12): error CS0246: Le nom de type ou d'espace de noms 'SudokuGrid' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)

(7,9): error CS0246: Le nom de type ou d'espace de noms 'DlxCustomized' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)

(7,43): error CS0246: Le nom de type ou d'espace de noms 'DlxCustomized' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)



Error: compilation error

### Analyse du solveur personnalisé

Le solveur `DancingLinkSolverWithCustomDlx` est un wrapper minimal autour de l'implementation personnalisée DLX.

| Composant | Responsabilité |
|-----------|----------------|
| **DlxCustomized** | Structure de données DLX et algorithme X |
| **Wrapper** | Interface ISudokuSolver pour integration |

**Points cles** :
1. Ce pattern de conception permet de separer l'algorithme de l'interface
2. La classe `DlxCustomized` gere toute la complexite de DLX
3. L'implementateur garde le controle complet sur les optimisations

> **Note technique** : Cette architecture permet de remplacer facilement l'implementation DLX sans modifier le reste du code.


## 5. Comparaison des Performances

Nous allons maintenant comparer les performances des deux approches sur des puzzles de différentes difficultés. Nous testerons les solveurs `Dancing Link Solver (DLXLib)` et `Dancing Link Solver (Customized)` sur des grilles de Sudoku de différentes difficultés (facile, moyenne, difficile).

### Résultats des Tests

Les temps d'exécution seront mesurés et comparés pour chaque solver et chaque niveau de difficulté.


In [6]:
using System.Diagnostics;
using System.Threading;
using System.Threading.Tasks;

var solvers = new List<(string Name, ISudokuSolver Solver)>
{
    ("Dancing Link Solver (DLXLib)", new DancingLinkSolver()),
    ("Dancing Link Solver (Customized)", new DancingLinkSolverWithCustomDlx())
};

var results = SudokuHelper.TestSolvers(solvers);

// Affichage des résultats
foreach (var result in results)
{
    Console.WriteLine($"{result.SolverName} | Difficulty: {result.Difficulty} | Time: {result.Time} ms | Status: {result.Status}");
}

SudokuHelper.DisplayResults(results);


(5,38): error CS0246: Le nom de type ou d'espace de noms 'ISudokuSolver' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)

(7,42): error CS0246: Le nom de type ou d'espace de noms 'DancingLinkSolver' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)

(8,46): error CS0246: Le nom de type ou d'espace de noms 'DancingLinkSolverWithCustomDlx' est introuvable (vous manque-t-il une directive using ou une référence d'assembly ?)

(11,15): error CS0103: Le nom 'SudokuHelper' n'existe pas dans le contexte actuel

(19,1): error CS0103: Le nom 'SudokuHelper' n'existe pas dans le contexte actuel



Error: compilation error

### Interpretation des resultats

Les resultats de la comparaison de performance montrent les differences entre l'approche DLXLib (bibliotheque) et l'implementation personnalisee.

| Approche | Avantages | Inconvenients |
|----------|-----------|---------------|
| **DLXLib** | Code concis, facile a maintenir, optimisee | Dependance externe |
| **Customized** | Complet maitrise du code, pas de dependance | Plus complexe a maintenir |

**Points cles** :
1. Les deux solveurs utilisent le meme algorithme de base (Algorithm X avec Dancing Links)
2. Les temps d'execution devraient etre comparables car l'algorithmique est identique
3. L'implementation customisee permet d'adapter le code a des besoins specifiques

> **Note technique** : Dancing Links est particulierement efficace pour les problemes de couverture exacte comme le Sudoku car il minimise les operations inutiles lors du backtracking.


---

<< [Sudoku-4-Z3](Sudoku-4-Z3.ipynb) | [Index](README.md) | [Sudoku-6-Infer](Sudoku-6-Infer.ipynb) >>

**Voir aussi** : 
- [Search-App-11-Picross](../Search/Applications/App-11-Picross.ipynb) - Application spectaculaire de DLX


## Exercice : Adapter Dancing Links pour le probleme des N-Reines

### Enonce

Le probleme des N-Reines consiste a placer N reines sur un echiquier N×N de sorte qu'aucune reine n'attaque une autre. Il peut etre formule comme un probleme de couverture exacte.

Adaptez le solveur Dancing Links pour resoudre le probleme des 8-Reines :
1. Definissez les contraintes (colonnes de la matrice DLX)
2. Definissez les lignes (placements possibles)
3. Comptez le nombre de solutions

### Indice

Pour le probleme des N-Reines :
- **Colonnes obligatoires** : chaque colonne du plateau doit avoir exactement une reine (N contraintes)
- **Colonnes optionnelles** : chaque diagonale peut avoir au plus une reine (2*(2N-1) contraintes)
- Chaque placement (ligne r, colonne c) couvre sa colonne, sa diagonale principale et sa diagonale secondaire

In [7]:
// EXERCICE : Probleme des N-Reines avec Dancing Links
// TODO: Implementez le comptage des solutions du probleme des 8-Reines
// en formulant le probleme comme une couverture exacte

// Parametres
int N = 8;

// Representation de la matrice DLX pour N-Reines :
// - N colonnes obligatoires : une reine par colonne (0..N-1)
// - 2*(2N-1) colonnes optionnelles : diagonales principales (N..3N-2) et secondaires (3N-1..5N-3)

// TODO : Implementer la classe NQueensDLX
// Elle doit :
// 1. Construire la matrice DLX avec les contraintes ci-dessus
// 2. Utiliser l'algorithme X pour trouver toutes les solutions
// 3. Retourner le nombre de solutions

public class NQueensDLX
{
    private int n;
    private int solutionCount = 0;
    
    public NQueensDLX(int size)
    {
        n = size;
    }
    
    public int CountSolutions()
    {
        // TODO: Implementer l'algorithme DLX pour compter les solutions
        // Etape 1: Initialiser la matrice avec les colonnes colonnes + diagonales
        // Etape 2: Ajouter une ligne pour chaque placement possible (r, c)
        // Etape 3: Lancer la recherche recursive et compter les solutions
        throw new NotImplementedException("A vous de jouer !");
    }
}

// Test de votre implementation
var nQueens = new NQueensDLX(N);
// int count = nQueens.CountSolutions();
// Console.WriteLine($"Nombre de solutions pour {N}-Reines : {count}");
// Console.WriteLine($"Attendu pour 8-Reines : 92");
Console.WriteLine("TODO: Implementez NQueensDLX.CountSolutions()");

TODO: Implementez NQueensDLX.CountSolutions()



(21,17): warning CS0414: Le champ 'NQueensDLX.solutionCount' est assigné, mais sa valeur n'est jamais utilisée

